In [ ]:
# this file digs into the address changes, now characterised by LSOAs.
#pip install ptitprince



In [ ]:
import pandas as pd
import json

In [ ]:
df_imd = pd.read_stata("S:\LLC_0002\data\stata_w_labs\CORE_lsoa11_geo_indicators_v0001_20220913.dta")

with open('living_periods_cohort.json', 'r') as json_a:
    living_periods_cohort = json.load(json_a)
    
    
with open('living_periods_nhsd.json', 'r') as json_b:
    living_periods_nhsd = json.load(json_b) 



In [ ]:
# only keep IMD Decile (for now)
columns_to_keep = ['lsoa11cd_e', 'imd2019_q10']
df_imd2019 = df_imd[columns_to_keep]
df_imd

In [ ]:
df_imd['lsoa11cd_e'] = df_imd['lsoa11cd_e'].astype(str)

In [ ]:
lsoa_to_imd = dict(zip(df_imd["lsoa11cd_e"], df_imd["imd2019_q10"]))

In [ ]:
for key, value in living_periods_cohort.items():
    if "lsoa11_e" in value:
        living_periods_cohort[key]["imd2019_q10"] = [lsoa_to_imd[lsoa] for lsoa in value["lsoa11_e"] if lsoa in lsoa_to_imd] 

In [ ]:
for key, value in living_periods_nhsd.items():
    if "lsoa11cd_e" in value:
        living_periods_nhsd[key]["imd2019_q10"] = [lsoa_to_imd[lsoa] for lsoa in value["lsoa11cd_e"] if lsoa in lsoa_to_imd] 

In [ ]:
living_periods_nhsd

In [ ]:
nhsd_results = []

for id_, values in living_periods_nhsd.items():
    imd_values = values['imd2019_q10']
    
    for i in range(len(imd_values) - 1):
        if imd_values[i] is None or imd_values[i + 1] is None:
            change = None
        else:
            change = imd_values[i+1] - imd_values[i]
        nhsd_results.append({
            "llc_0002_stud_id": id_,
            "from_imd": imd_values[i],
            "to_imd": imd_values[i+1],
            "imd_change": change,
            "imd_number": i + 1
        })
        
nhsd_imdchange = pd.DataFrame(nhsd_results)

nhsd_imdchange['total_address_changes'] = nhsd_imdchange.groupby('llc_0002_stud_id')['imd_number'].transform(
    lambda x:x.notna().sum()
)

nhsd_imdchange

In [ ]:
# nhsd IMD change matrix

import numpy as np

transition_counts = pd.crosstab(nhsd_imdchange['from_imd'], nhsd_imdchange['to_imd'])
transition_matrix_nhsd = transition_counts.div(transition_counts.sum(axis = 1), axis = 0)
transition_matrix_nhsd




In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize = (8,6))
sns.heatmap(transition_matrix_nhsd, annot = True, cmap = "Blues", fmt = " .2f", cbar = True)
plt.title("NHSD: Transition Matrix")
plt.ylabel("From IMD")
plt.xlabel("To IMD")
plt.show()

In [ ]:
# distribution of changes

valid_changes = nhsd_imdchange['imd_change'].dropna()

valid_changes


In [ ]:
# histogram
sns.histplot(valid_changes, bins = 18, kde = False, color = 'green')
plt.xlim(-9,9)
plt.xticks(range(-9,10))
plt.title("distribution of IMD Changes in NHSD LSOAs")
plt.show()

In [ ]:
filtered_df = nhsd_imdchange.dropna(subset = ['imd_change','from_imd'])

g = sns.FacetGrid(filtered_df, col = "from_imd", col_wrap = 3, sharey = False, sharex = True, height = 4)
g.map(sns.histplot, "imd_change", bins = 18, kde = False, color = 'skyblue')

for ax in g.axes.flat:
    ax.set_xlabel("IMD Change")
    
g.set_axis_labels("", "Frequency")
g.fig.suptitle("Distribution of IMD Change by `from_imd'", y = 1.03)
    

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

for imd_value in sorted(filtered_df['from_imd'].unique()):
    subset = filtered_df[filtered_df['from_imd'] == imd_value] 
    sns.kdeplot(subset['imd_change'], label = f"from IMD = {imd_value}", fill = True)

plt.xlim(-9,9)
plt.xticks(range(-9,10))

plt.legend(title = "From IMD")
plt.tight_layout()
plt.show()

In [ ]:
import ptitprince as pt

pal = "Set2"
palette = sns.color_palette(pal)
f, ax = plt.subplots(figsize = (22, 10))
dy = "from_imd"; dx = "imd_change"


ax = pt.half_violinplot(x = dx, y = dy, data = filtered_df[filtered_df["from_imd"].isin([1.0,2.0,3.0,4.0,5.0])], palette = pal, bw = .2, cut = 0.,
                       scale = "area", width = .6, inner = None, orient = 'h')

ax = sns.boxplot(data = filtered_df[filtered_df["from_imd"].isin([1.0,2.0,3.0,4.0,5.0])], y = 'from_imd', x = 'imd_change',  palette = pal, width = .15, zorder = 10,
                showfliers = True, whiskerprops = {'linewidth':2, "zorder": 10},
                saturation = 1, orient = 'h',
           )

plt.xlim(-9,9)
plt.xticks(range(-9,10))

plt.legend(title = "From IMD")
plt.tight_layout()
plt.show()


In [ ]:
pal = "Set2"
palette = sns.color_palette(pal)
f, ax = plt.subplots(figsize = (22, 10))
dy = "from_imd"; dx = "imd_change"


ax = pt.half_violinplot(x = dx, y = dy, data = filtered_df[filtered_df["from_imd"].isin([6.0,7.0,8.0,9.0,10.0])], palette = pal, bw = .2, cut = 0.,
                       scale = "area", width = .6, inner = None, orient = 'h')

ax = sns.boxplot(data = filtered_df[filtered_df["from_imd"].isin([6.0,7.0,8.0,9.0,10.0])], y = 'from_imd', x = 'imd_change',  palette = pal, width = .15, zorder = 10,
                showfliers = True, whiskerprops = {'linewidth':2, "zorder": 10},
                saturation = 1, orient = 'h',
           )

plt.xlim(-9,9)
plt.xticks(range(-9,10))

plt.legend(title = "From IMD")
plt.tight_layout()
plt.show()


In [ ]:
cohort_results = []

for id_, values in living_periods_cohort.items():
    imd_values = values['imd2019_q10']
    
    for i in range(len(imd_values) - 1):
        if imd_values[i] is None or imd_values[i + 1] is None:
            change = None
        else:
            change = imd_values[i+1] - imd_values[i]
        cohort_results.append({
            "llc_0002_stud_id": id_,
            "from_imd": imd_values[i],
            "to_imd": imd_values[i+1],
            "imd_change": change,
            "imd_number": i + 1
        })
        
cohort_imdchange = pd.DataFrame(cohort_results)

"""cohort_imdchange['total_address_changes'] = cohort_imdchange.groupby('llc_0002_stud_id')['imd_number'].transform(
    lambda x:x.notna().sum()
)"""

cohort_imdchange

In [ ]:

# distribution of changes

valid_changes = cohort_imdchange['imd_change'].dropna()

# histogram
sns.histplot(valid_changes, bins = 10, kde = False, color = 'green')
plt.title("distribution of IMD Changes in Cohorts")
plt.xticks(range(-9,10))
plt.show()


In [ ]:

transition_counts = pd.crosstab(cohort_imdchange['from_imd'], cohort_imdchange['to_imd'])
transition_matrix_cohort = transition_counts.div(transition_counts.sum(axis = 1), axis = 0)
transition_matrix_cohort

In [ ]:

plt.figure(figsize = (8,6))
sns.heatmap(transition_matrix_cohort, annot = True, cmap = "Blues", fmt = " .2f", cbar = True)
plt.title("Cohort: Transition Matrix")
plt.ylabel("From IMD")
plt.xlabel("To IMD")
plt.show()

In [ ]:
filtered_df = cohort_imdchange.dropna(subset = ['imd_change','from_imd'])
plt.figure(figsize=(10,6))

for imd_value in sorted(filtered_df['from_imd'].unique()):
    subset = filtered_df[filtered_df['from_imd'] == imd_value] 
    sns.kdeplot(subset['imd_change'], label = f"from IMD = {imd_value}", fill = True)

plt.xlim(-9,9)
plt.xticks(range(-9,10))

plt.legend(title = "From IMD")
plt.tight_layout()
plt.show()

In [ ]:
filtered_df = cohort_imdchange.dropna(subset = ['imd_change','from_imd'])
pal = "Set2"
palette = sns.color_palette(pal)
f, ax = plt.subplots(figsize = (22, 10))
dy = "from_imd"; dx = "imd_change"


ax = pt.half_violinplot(x = dx, y = dy, data = filtered_df[filtered_df["from_imd"].isin([1.0,2.0,3.0,4.0,5.0])], palette = pal, bw = .2, cut = 0.,
                       scale = "area", width = .6, inner = None, orient = 'h')

ax = sns.boxplot(data = filtered_df[filtered_df["from_imd"].isin([1.0,2.0,3.0,4.0,5.0])], y = 'from_imd', x = 'imd_change',  palette = pal, width = .15, zorder = 10,
                showfliers = True, whiskerprops = {'linewidth':2, "zorder": 10},
                saturation = 1, orient = 'h',
           )

plt.xlim(-9,9)
plt.xticks(range(-9,10))

plt.legend(title = "From IMD")
plt.tight_layout()
plt.show()


In [ ]:
pal = "Set2"
palette = sns.color_palette(pal)
f, ax = plt.subplots(figsize = (22, 10))
dy = "from_imd"; dx = "imd_change"


ax = pt.half_violinplot(x = dx, y = dy, data = filtered_df[filtered_df["from_imd"].isin([6.0,7.0,8.0,9.0,10.0])], palette = pal, bw = .2, cut = 0.,
                       scale = "area", width = .6, inner = None, orient = 'h')

ax = sns.boxplot(data = filtered_df[filtered_df["from_imd"].isin([6.0,7.0,8.0,9.0,10.0])], y = 'from_imd', x = 'imd_change',  palette = pal, width = .15, zorder = 10,
                showfliers = True, whiskerprops = {'linewidth':2, "zorder": 10},
                saturation = 1, orient = 'h',
           )

plt.xlim(-9,9)
plt.xticks(range(-9,10))

plt.legend(title = "From IMD")
plt.tight_layout()
plt.show()


In [ ]:
# compare the matrix-es

diff_matrix = transition_matrix_nhsd - transition_matrix_cohort
plt.figure(figsize = (8,6))
cmap = sns.diverging_palette(250, 10, as_cmap=True)
sns.heatmap(diff_matrix, annot = True, cmap = cmap , center = 0, fmt = " .2f", cbar = True,
           xticklabels=transition_matrix_nhsd.columns, yticklabels=transition_matrix_nhsd.index)
plt.xlabel("To IMD")
plt.ylabel("From IMD")
plt.title("Comparing 2 matrices")
plt.show()


In [ ]:
# normalised transition matrices - probability 
transition_matrix_nhsd_norm = transition_matrix_nhsd.div(transition_matrix_nhsd.sum(axis = 1), axis = 0)
transition_matrix_cohort_norm = transition_matrix_cohort.div(transition_matrix_cohort.sum(axis = 1), axis = 0)


In [ ]:
transition_matrix_nhsd_norm



In [ ]:
diagonal_nhsd = np.diagonal(transition_matrix_nhsd_norm)
mean_diagonal_nhsd = diagonal_nhsd.mean()
mean_diagonal_nhsd

In [ ]:
transition_matrix_cohort_norm

In [ ]:
diagonal_cohort = np.diagonal(transition_matrix_cohort_norm)
mean_diagonal_cohort = diagonal_cohort.mean()
mean_diagonal_cohort

In [ ]:
frobenius_norm = np.linalg.norm(diff_matrix.to_numpy())
frobenius_norm


In [ ]:
from scipy.special import kl_div

kl_divergence = np.sum(kl_div(transition_matrix_nhsd.to_numpy(), transition_matrix_cohort.to_numpy()))
kl_divergence

In [ ]:
### This section & below are not used in the article. These code calculates the proportion of missed match IMD using each method. ###

# start with N-1 

from datetime import datetime
from collections import defaultdict

def calculate_imd_index(dict_a, dict_b):
    mismatch_results = []
    censor_date = datetime.strptime('2023-04-06', '%Y-%m-%d')
    for unique_id in dict_a:
        if unique_id in dict_b:
            info_a = dict_a[unique_id]
            info_b = dict_b[unique_id]
            # a = n-1, b = median
            address_map_a = {address: (start_a, end_a, imd_a)
                             for address, start_a, end_a, imd_a in zip(info_a['lsoa11_e'],
                                                                                     info_a['start_date'], 
                                                                                     info_a['end_date_method_n1'], 
                                                                                     info_a['imd2019_q10'])
                            } 
            address_map_b = {address: (start_b, end_b, imd_b) 
                             for address, start_b, end_b, imd_b in zip(info_b['lsoa11cd_e'], 
                                                                                     info_b['start_date'], 
                                                                                     info_b['end_date_method_n1'],  
                                                                                     info_b['imd2019_q10'])
                            } 

            for address_a, (start_a, end_a, imd_a) in address_map_a.items():
                for address_b, values_b in address_map_b.items():
                    # get dates for both methods in both dictionaries
                    start_b, end_b, imd_b = values_b
                    #parse dates if strings
                    if isinstance(start_a, str):
                        start_a = datetime.strptime(start_a, '%Y-%m-%d')
                    if isinstance(end_a, str):    
                        end_a = datetime.strptime(end_a, '%Y-%m-%d')
                    if isinstance(start_b, str):
                        start_b = datetime.strptime(start_b, '%Y-%m-%d')
                    if isinstance(end_b, str):
                        end_b = datetime.strptime(end_b, '%Y-%m-%d')
                        
                    if end_a < start_a:
                        end_a = censor_date
                    if end_b < start_b:
                        end_b = censor_date

                    # initialise variable
                    mismatch_period = 0
                    imd_difference = 0
                    
                    # case 1 - n-1
                    if address_a != address_b and end_b < end_a and end_b > start_a and start_b < start_a:
                        mismatch_period = (end_b - start_a).days
                        imd_difference = (imd_b - imd_a)

                    # case 2 - n-1
                    elif address_a != address_b and end_b > end_a and start_b < end_a and start_b > start_a:
                        mismatch_period = (end_a - start_b).days
                        imd_difference = imd_b - imd_a
                        
                    # case 3 - n-1
                    elif address_a != address_b and start_b < start_a and end_b > end_a:
                        mismatch_period = (end_a - start_a).days
                        imd_difference = imd_b - imd_a
                    # case 4 - n-1
                    elif address_a != address_b and start_b > start_a and end_b < end_a:
                        mismatch_period = (end_b - start_b).days
                        imd_difference = imd_b - imd_a

                    total_period = (end_a - start_a).days    
                    if mismatch_period > 0 and total_period > 0:
                        mismatch_index = imd_difference * (mismatch_period/total_period)
                    else:
                        mismatch_index = 0

                    mismatch_results.append({
                        'llc_0002_stud_id': unique_id,
                        'lsoa_cohort': address_a,
                        'lsoa_nhsd': address_b,
                        'starta':start_a,
                        'enda': end_a,
                        'startb':start_b,
                        'endb':end_b,
                        'imd_cohort': imd_a,
                        'imd_nhsd': imd_b,
                        'mismatch_period': mismatch_period,
                        'imd_difference': imd_difference,
                        
                    })    
    return mismatch_results
                


In [ ]:
mismatch_results_n1 = calculate_imd_index(living_periods_cohort, living_periods_nhsd)

In [ ]:
df_mismatch_results_n1 = pd.DataFrame(mismatch_results_n1)

In [ ]:
df_mismatch_results_n1_filtered = df_mismatch_results_n1[df_mismatch_results_n1['mismatch_period'] != 0]

In [ ]:
df_mismatch_results_n1_filtered

In [ ]:
# wrong end_date using n-1 method, replace with end_date
df_mismatch_results_n1_filtered[df_mismatch_results_n1_filtered['mismatch_period'] < 0]

In [ ]:
# extract total days from cohort
def calculate_days_difference(data):
    results = []
    for ID, dates in data.items():
        start_dates = [datetime.strptime(date, '%Y-%m-%d') for date in dates['start_date']]
        end_dates = [datetime.strptime(date, '%Y-%m-%d') for date in dates['end_date_method_n1']]
        days_difference = (max(end_dates) - min(start_dates)).days
        results.append({'llc_0002_stud_id': ID, 'days_diff': days_difference})
        
    return pd.DataFrame(results)

results_df = calculate_days_difference(living_periods_cohort)


In [ ]:
results_df

In [ ]:
df_mismatch_n1_combined = df_mismatch_results_n1_filtered.merge(results_df, 
                                                                on = 'llc_0002_stud_id', 
                                                                how = 'left', 
                                                                suffixes = ('_1', '_2'))

In [ ]:
df_mismatch_n1_combined

In [ ]:
df_mismatch_n1_combined['proportion'] = df_mismatch_n1_combined['mismatch_period']/df_mismatch_n1_combined['days_diff']

In [ ]:
df_mismatch_n1_combined

In [ ]:
df_mismatch_n1_combined[df_mismatch_n1_combined['llc_0002_stud_id']=="110653087183136"]

In [ ]:
# calculate index
df_mismatch_n1_combined['imd_index'] = df_mismatch_n1_combined['proportion'] * df_mismatch_n1_combined['imd_difference']


In [ ]:
df_mismatch_n1_combined[df_mismatch_n1_combined['proportion'] < 0]

In [ ]:
df_total_n1 = df_mismatch_n1_combined.groupby(['llc_0002_stud_id'], as_index = False).sum()
columns_to_keep = ['llc_0002_stud_id', 'imd_index']
df_total_n1 = df_total_n1[columns_to_keep]

In [ ]:
df_total_n1.describe()

In [ ]:
df_total_n1

In [ ]:
import numpy as np
plt.figure(figsize = (10,6))
sns.histplot(df_total_n1['imd_index'], kde = True, bins = 50, color = 'blue', alpha = 0.7)

p25,p50,p75 = np.percentile(df_total_n1['imd_index'], [25,50,75])
plt.axvline(x = p25, color = 'purple', linestyle = '--', label = f'P25: {p25:.2f}')
plt.axvline(x = p50, color = 'orange', linestyle = '--', label = f'P50: {p50:.2f}')
plt.axvline(x = p75, color = 'cyan', linestyle = '--', label = f'P75: {p75:.2f}')


plt.grid(True)
plt.legend()
plt.show()

In [ ]:
df_total_n1['imd_index'].describe()


In [ ]:
## repeat analysis for median 

from datetime import datetime
from collections import defaultdict

def calculate_imd_index(dict_a, dict_b):
    mismatch_results = []
    censor_date = datetime.strptime('2023-04-06', '%Y-%m-%d')
    for unique_id in dict_a:
        if unique_id in dict_b:
            info_a = dict_a[unique_id]
            info_b = dict_b[unique_id]
            # a = n-1, b = median
            address_map_a = {address: (start_a, end_a, imd_a)
                             for address, start_a, end_a, imd_a in zip(info_a['lsoa11_e'],
                                                                                     info_a['start_date_median'], 
                                                                                     info_a['end_date_method_med'], 
                                                                                     info_a['imd2019_q10'])
                            } 
            address_map_b = {address: (start_b, end_b, imd_b) 
                             for address, start_b, end_b, imd_b in zip(info_b['lsoa11cd_e'], 
                                                                                     info_b['start_date_median'], 
                                                                                     info_b['end_date_method_med'],  
                                                                                     info_b['imd2019_q10'])
                            } 

            for address_a, (start_a, end_a, imd_a) in address_map_a.items():
                for address_b, values_b in address_map_b.items():
                    # get dates for both methods in both dictionaries
                    start_b, end_b, imd_b = values_b
                    #parse dates if strings
                    if isinstance(start_a, str):
                        start_a = datetime.strptime(start_a, '%Y-%m-%d')
                    if isinstance(end_a, str):    
                        end_a = datetime.strptime(end_a, '%Y-%m-%d')
                    if isinstance(start_b, str):
                        start_b = datetime.strptime(start_b, '%Y-%m-%d')
                    if isinstance(end_b, str):
                        end_b = datetime.strptime(end_b, '%Y-%m-%d')
                        
                    if end_a < start_a:
                        end_a = censor_date
                    if end_b < start_b:
                        end_b = censor_date

                    # initialise variable
                    mismatch_period = 0
                    imd_difference = 0
                    
                    # case 1 - n-1
                    if address_a != address_b and end_b < end_a and end_b > start_a and start_b < start_a:
                        mismatch_period = (end_b - start_a).days
                        imd_difference = (imd_b - imd_a)

                    # case 2 - n-1
                    elif address_a != address_b and end_b > end_a and start_b < end_a and start_b > start_a:
                        mismatch_period = (end_a - start_b).days
                        imd_difference = imd_b - imd_a
                        
                    # case 3 - n-1
                    elif address_a != address_b and start_b < start_a and end_b > end_a:
                        mismatch_period = (end_a - start_a).days
                        imd_difference = imd_b - imd_a
                    # case 4 - n-1
                    elif address_a != address_b and start_b > start_a and end_b < end_a:
                        mismatch_period = (end_b - start_b).days
                        imd_difference = imd_b - imd_a

                    total_period = (end_a - start_a).days    
                    if mismatch_period > 0 and total_period > 0:
                        mismatch_index = imd_difference * (mismatch_period/total_period)
                    else:
                        mismatch_index = 0

                    mismatch_results.append({
                        'llc_0002_stud_id': unique_id,
                        'lsoa_cohort': address_a,
                        'lsoa_nhsd': address_b,
                        'starta':start_a,
                        'enda': end_a,
                        'startb':start_b,
                        'endb':end_b,
                        'imd_cohort': imd_a,
                        'imd_nhsd': imd_b,
                        'mismatch_period': mismatch_period,
                        'imd_difference': imd_difference,
                        
                    })    
    return mismatch_results
                



In [ ]:
mismatch_results_med = calculate_imd_index(living_periods_cohort, living_periods_nhsd)
df_mismatch_results_med = pd.DataFrame(mismatch_results_med)
df_mismatch_results_med_filtered = df_mismatch_results_med[df_mismatch_results_med['mismatch_period'] != 0]

In [ ]:
df_mismatch_results_med_filtered

In [ ]:
df_mismatch_med_combined = df_mismatch_results_med_filtered.merge(results_df, 
                                                                on = 'llc_0002_stud_id', 
                                                                how = 'left', 
                                                                suffixes = ('_1', '_2'))

In [ ]:
df_mismatch_med_combined['proportion'] = df_mismatch_med_combined['mismatch_period']/df_mismatch_med_combined['days_diff']

In [ ]:
df_mismatch_med_combined['imd_index'] = df_mismatch_med_combined['proportion'] * df_mismatch_med_combined['imd_difference']
df_total_med = df_mismatch_med_combined.groupby(['llc_0002_stud_id'], as_index = False).sum()
columns_to_keep = ['llc_0002_stud_id', 'imd_index']
df_total_med = df_total_med[columns_to_keep]

In [ ]:
df_total_n1.describe()

In [ ]:
df_total_med

In [ ]:
plt.figure(figsize = (10,6))
sns.histplot(df_total_n1['imd_index'], kde = False, label = "N-1",bins = np.arange(-5,5.5,0.2), color = 'blue', alpha = 0.5)
sns.histplot(df_total_med['imd_index'], kde = False, label = "Median", bins = np.arange(-5,5.5,0.2), color = 'red', alpha = 0.5)

plt.xticks(np.arange(-5,5.5,0.5))
plt.grid(True)
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles=handles, labels = labels, title='Compare methods', loc = 'upper right')
plt.show()

In [ ]:
df_demo = pd.read_csv(r"S:\LLC_0002\lamj\Datasets\Multiple-Source_Harmonisation\demographics.csv")
df_demo['gender_or_sex'] = df_demo['sex'].replace("Missing", pd.NA).fillna(df_demo['gender'])
df_demo = df_demo.drop(columns = ["gender","sex"])
df_demo


In [ ]:
# N-1
df_demo_n1 = pd.concat([df_total_n1, df_demo],axis = 1, join = 'inner')

In [ ]:
df_demo_n1

In [ ]:
plt.figure(figsize = (10,6))
sns.kdeplot(
    data = df_demo_n1,
    x= 'imd_index',
    hue = 'age',
    fill = True,
    palette = "Set2",
    alpha= 0.6,
    common_norm = False
)


plt.title("Density Plot of Mismatch Index by age", fontsize= 14)
plt.legend(title = "age",
          loc = "upper right",
          frameon = True,
          shadow = True)
plt.grid(True)
plt.show()

In [ ]:
df_demo_med = pd.concat([df_total_med, df_demo],axis = 1, join = 'inner')